# OneWaterAI · Ko Sichang
### Environmental × MST × eDNA × Machine Learning

A complete three-page Gradio application, revised from your 1,069-line source. **All measurements, labels, scores and model evaluation remain synthetic and unvalidated.** The Ridge/MLP models, score formula, time split, lag logic, 30-day scenarios and seven role workflows are preserved.

## Run in Google Colab
1. Upload this notebook at [Google Colab](https://colab.research.google.com/) → **File → Upload notebook**. Use a fresh CPU runtime.
2. Run the dependency cell, choose **Runtime → Restart session**, then run only the complete application cell. The restart is required after replacing Colab's NumPy stack.
3. Open the printed **temporary Gradio URL** (ending `?__theme=light`) and share it. Visitors do not need Colab. Keep this runtime running; the link stops when the app/runtime stops.

No API keys or separate hosting are required. Internet is needed for dependencies, public sharing and map tiles. The three pages and dropdown site selector work even when map tiles fail. Downloads and submissions are temporary; download receipts before closing. Avoid personal/sensitive information.

## Optional authentic GIS
Only the GIS screenshot was supplied; **no geographic layers are fabricated or reconstructed from it.** The default map uses the original simulated point coordinates on a third-party basemap. Use **Dashboard → Optional GIS** to upload GeoJSON and/or GeoTIFF files. Invalid layers are rejected with a visible reason while valid layers can load independently.

To preload files, put them in `/content/gis/` before running the application cell. Paths/names and the R export guide are in the accompanying `GIS_GUIDE.md`. Preloaded GIS is shared at app startup; UI uploads apply to the current browser session.

Needed from the original analysis: R script, modeled B01–B12 catchments and flow-path vectors, slope/groundwater-screening GeoTIFFs with CRS/NoData, provenance and any surveyed station coordinates. A catchment is not a sampling location; B-ID risk joining is illustrative until verified. **Groundwater screening is an unvalidated slope-only proxy, not measured groundwater availability.**


In [ ]:
# Compatible with current Google Colab Python 3.13.
# NumPy 2.2.6 fails while importing numpy.strings on this runtime.
%pip -q install --upgrade --force-reinstall gradio==5.49.1 numpy==2.3.3 pandas==2.3.3 scipy==1.16.2 scikit-learn==1.7.2 plotly==6.3.0 rasterio==1.4.3 pillow==11.3.0 shapely==2.1.1 huggingface-hub==0.35.3

# REQUIRED after this cell finishes: Runtime > Restart session.
# Then run only the complete application cell below. This prevents Colab from
# retaining old NumPy modules after replacing its preinstalled scientific stack.


In [ ]:
import csv
import uuid
import shutil
import threading
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.exceptions import ConvergenceWarning


# ─────────────────────── Configuration ───────────────────────

SEED = 20261016
IDS = [f"B{i:02d}" for i in range(1, 13)]
DEFAULT = "B05"

HUMAN = ["HF183", "crAssphage", "PMMoV"]
ANIMAL = ["C16S", "qC160F"]
BIO = ["MiFish", "MollCOI253", "Ceph16S", "Crust16S"]
PATHOGEN = ["Norovirus GI", "Norovirus GII", "VP28", "YHV-1", "AHPND", "NNV"]

MICROBES = HUMAN + ANIMAL + BIO + PATHOGEN
ENV = ["Rainfall", "Temperature", "DO", "pH"]
EDIT = ENV + MICROBES
LAGS = ["rain_lag1", "rain_lag3", "rain_prev7", "dry_before", "first_rain"]
NUM = EDIT + LAGS
XCOLS = ["site_id"] + NUM

GROUPS = [
    ("ตัวบ่งชี้ที่เกี่ยวข้องกับอุจจาระมนุษย์", HUMAN),
    ("ตัวบ่งชี้สัตว์บก/สัตว์ปีก — candidate assays", ANIMAL),
    ("ความหลากหลายทางชีวภาพ / eDNA", BIO),
    ("เป้าหมายเชื้อก่อโรค — แยกจาก MST", PATHOGEN),
]

RANGES = {
    "Rainfall": (0, 120),
    "Temperature": (22, 36),
    "DO": (0.5, 12),
    "pH": (6, 9),
}
ENV_LABELS = [
    "Rainfall — ฝน (mm)",
    "Temperature — อุณหภูมิน้ำ (°C)",
    "DO — ออกซิเจนละลาย (mg/L)",
    "pH",
]

CATS = ["ต่ำ · สาธิต", "ปานกลาง · สาธิต", "สูง · สาธิต"]
COLORS = dict(zip(CATS, ["#16877e", "#c29539", "#bc5651"]))
SCENARIOS = ["บล็อกข้อมูลย้อนหลัง", "ทดสอบฝนแรกหลังแล้ง 7 วัน"]

OUT = Path.cwd() / "onewaterai_outputs"
OUT.mkdir(exist_ok=True)
SAVE_LOCK = threading.Lock()


def category(score):
    return CATS[0] if score < 35 else CATS[1] if score < 65 else CATS[2]


def action(score):
    if score >= 65:
        return "ส่งผู้รับผิดชอบทบทวนข้อมูลและวางแผนตรวจยืนยัน"
    if score >= 35:
        return "ทบทวนข้อมูลที่ยังขาดและกำหนดการติดตาม"
    return "บันทึกข้อมูลและติดตามตามแผน — ยังไม่ยืนยันความปลอดภัย"


# ─────────────────────── Synthetic daily data ───────────────────────

def features(frame):
    """สร้าง lag จากฝนก่อนหน้าในจุดเดียวกันเท่านั้น"""
    d = frame.copy().sort_values(["site_id", "date"]).reset_index(drop=True)
    g = d.groupby("site_id")["Rainfall"]

    d["rain_lag1"] = g.shift(1).fillna(0)
    d["rain_lag3"] = g.shift(3).fillna(0)
    d["rain_prev7"] = g.transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).sum()
    ).fillna(0)

    d["dry_before"] = 0
    for _, part in d.groupby("site_id"):
        count, counts = 0, []
        for rain in part["Rainfall"]:
            counts.append(count)
            count = count + 1 if rain < 1 else 0
        d.loc[part.index, "dry_before"] = counts

    d["first_rain"] = (
        (d["Rainfall"] >= 10) & (d["dry_before"] >= 5)
    ).astype(int)
    return d


def generate_data():
    rng = np.random.default_rng(SEED)
    dates = pd.date_range(end="2026-10-15", periods=450)
    season = np.sin(2 * np.pi * np.arange(len(dates)) / 365)

    regional_rain = np.where(
        rng.random(len(dates)) < 0.28 + 0.12 * season,
        rng.gamma(1.7, 13, len(dates)),
        0,
    )

    frames = []
    for i, site in enumerate(IDS):
        d = pd.DataFrame({"date": dates, "site_id": site})
        d["site_name"] = f"จุดสาธิตชายฝั่ง {i+1:02d}"

        # พิกัดสมมติรอบเกาะสีชัง ไม่ใช่สถานีตรวจวัดจริง
        angle = 2 * np.pi * i / 12
        d["latitude"] = 13.153 + 0.018 * np.sin(angle)
        d["longitude"] = 100.811 + 0.012 * np.cos(angle)

        human_base = 10 + 75 * i / 11
        animal_base = rng.uniform(8, 75)
        aquatic_base = rng.uniform(8, 85)

        d["Rainfall"] = np.clip(
            regional_rain * rng.uniform(0.8, 1.2), 0, 120
        )
        d["Temperature"] = np.clip(
            29 + 2 * season + rng.normal(0, 0.7, len(d)), 24, 34
        )
        d["DO"] = np.clip(
            8.8 - 0.05 * human_base - 0.02 * aquatic_base
            - 0.12 * (d.Temperature - 26)
            + 0.012 * d.Rainfall
            + rng.normal(0, 0.5, len(d)),
            1, 10,
        )
        d["pH"] = np.clip(
            8 + 0.15 * season - 0.003 * d.Rainfall
            + rng.normal(0, 0.13, len(d)),
            6.6, 8.8,
        )

        d = features(d)
        delayed = (
            0.6 * np.tanh(d.rain_lag1 / 20)
            + 0.4 * np.tanh(d.rain_lag3 / 20)
        )
        human = (
            human_base + 14 * delayed
            + 18 * d.first_rain * np.tanh(d.Rainfall / 20)
            - 15 * np.tanh(d.Rainfall / 40)
        )
        animal = animal_base + 9 * delayed - 9 * np.tanh(d.Rainfall / 40)
        aquatic = aquatic_base + 9 * season + 4 * np.maximum(0, 5 - d.DO)

        for name in HUMAN:
            d[name] = np.clip(human + rng.normal(0, 8, len(d)), 0, 100)
        for name in ANIMAL:
            d[name] = np.clip(animal + rng.normal(0, 9, len(d)), 0, 100)
        for name in BIO:
            d[name] = np.clip(
                65 - 0.2 * aquatic + 10 * season
                + rng.normal(0, 12, len(d)), 0, 100
            )
        for name in PATHOGEN:
            base = human if name.startswith("Norovirus") else aquatic
            d[name] = np.clip(base + rng.normal(0, 10, len(d)), 0, 100)

        # สูตรสาธิตที่แต่งขึ้น ไม่ใช่สมการความเสี่ยงที่ผ่าน validation
        h = d[HUMAN].mean(axis=1) / 100
        a = d[ANIMAL].mean(axis=1) / 100
        ph = d[PATHOGEN[:2]].mean(axis=1) / 100
        pa = d[PATHOGEN[2:]].mean(axis=1) / 100
        stress = np.clip((6 - d.DO) / 5, 0, 1)

        # ฝนมีทั้งพจน์บวกและลบ ขึ้นกับบริบทจำลอง
        rain_effect = (
            0.08 * d.first_rain * np.tanh(d.Rainfall / 30)
            + 0.10 * np.tanh(d.rain_prev7 / 60) * (1 - d.DO / 10)
            - 0.10 * np.tanh(d.Rainfall / 40) * (d.DO / 10)
        )
        score = 100 * (
            0.03 + 0.20*h + 0.08*a + 0.26*ph + 0.15*pa
            + 0.16*stress + 0.04*abs(d.pH - 8)/1.5
            + 0.02*np.maximum(d.Temperature - 30, 0)/5
            + rain_effect
        )

        # Biodiversity/eDNA ไม่มีน้ำหนักเพิ่มคะแนนโดยตรง
        d["risk_score"] = np.clip(
            score + rng.normal(0, 2.2, len(d)), 0, 100
        )
        d["data_status"] = "SYNTHETIC_ONLY"
        frames.append(d)

    return pd.concat(frames, ignore_index=True)


DATA = generate_data()
LATEST = (
    DATA.sort_values("date").groupby("site_id").tail(1)
    .set_index("site_id").loc[IDS]
)
LATEST["risk_category"] = LATEST.risk_score.map(category)
LATEST["recommended_action"] = LATEST.risk_score.map(action)
DATA.to_csv(OUT / "synthetic_daily.csv", index=False, encoding="utf-8-sig")


# ─────────────────────── Train comparable models ───────────────────────

# ตัดช่วงเริ่มต้นที่ยังมีประวัติ lag ไม่ครบ
eligible = DATA[DATA.groupby("site_id").cumcount() >= 7].copy()
dates = np.sort(eligible.date.unique())
CUTOFF = pd.Timestamp(dates[int(0.70 * len(dates))])

# ทุกจุดใช้เส้นแบ่งวันที่เดียวกัน ป้องกันการสุ่มวันอนาคตเข้า train
TRAIN = eligible[eligible.date < CUTOFF]
TEST = eligible[eligible.date >= CUTOFF]
TRAIN_MEANS = TRAIN[NUM].mean()

MODELS, evaluation = {}, []
estimators = [
    ("Simple Regression (Ridge)", Ridge(alpha=5)),
    (
        "Neural Network (MLPRegressor)",
        MLPRegressor(
            hidden_layer_sizes=(32, 16),
            max_iter=400,
            early_stopping=False,
            n_iter_no_change=25,
            random_state=SEED,
        ),
    ),
]

for name, estimator in estimators:
    preprocess = ColumnTransformer([
        ("numeric", StandardScaler(), NUM),
        (
            "site",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["site_id"],
        ),
    ])
    model = TransformedTargetRegressor(
        regressor=Pipeline([
            ("preprocess", preprocess),
            ("model", estimator),
        ]),
        transformer=StandardScaler(),
    )

    with warnings.catch_warnings(record=True) as records:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(TRAIN[XCOLS], TRAIN.risk_score)

    predicted = np.clip(model.predict(TEST[XCOLS]), 0, 100)
    status = (
        "ถึง max_iter; ควรทบทวน convergence"
        if any(issubclass(w.category, ConvergenceWarning) for w in records)
        else "จบการฝึก"
    )
    evaluation.append([
        name,
        r2_score(TEST.risk_score, predicted),
        mean_absolute_error(TEST.risk_score, predicted),
        np.sqrt(mean_squared_error(TEST.risk_score, predicted)),
        status,
    ])
    MODELS[name] = model

MODEL_NAMES = list(MODELS)
METRICS = pd.DataFrame(
    evaluation, columns=["โมเดล", "R²", "MAE", "RMSE", "สถานะ"]
)


def predict(model_name, rows):
    return np.clip(MODELS[model_name].predict(rows[XCOLS]), 0, 100)


# ─────────────────────── Dashboard helpers ───────────────────────

def styled(fig, title, height=380):
    fig.update_layout(title=dict(text=title,font=dict(size=14,color='#193b48')),
        template='plotly_white',height=height,
        font=dict(family='Noto Sans Thai, Tahoma, Arial',size=11,color='#607f8c'),
        paper_bgcolor='white',plot_bgcolor='white',
        margin=dict(l=35,r=25,t=50,b=40),
        legend=dict(orientation='h',y=-.18,x=0),
        modebar_remove=['lasso2d','select2d','autoScale2d'],hoverlabel=dict(bgcolor='white'))
    fig.update_xaxes(gridcolor='#edf2f2',zerolinecolor='#e1eaea')
    fig.update_yaxes(gridcolor='#edf2f2',zerolinecolor='#e1eaea')
    return fig


def indicator_table(row, names, environmental=False):
    units = dict(zip(ENV, ["mm", "°C", "mg/L", "ไม่มีหน่วย"]))
    return pd.DataFrame({
        "Indicator": names,
        "ค่า": [round(float(row[n]), 2) for n in names],
        "หน่วย": [
            units[n] if environmental else "ดัชนีจำลอง 0–100"
            for n in names
        ],
    })


def home_view(site):
    row = LATEST.loc[site]
    hist = DATA[DATA.site_id == site]
    text = (
        f"### {site} — {row.site_name}\n"
        f"**คะแนน {row.risk_score:.1f}/100 · {category(row.risk_score)}**  \n"
        f"{action(row.risk_score)}  \n"
        f"วันที่ข้อมูลจำลอง: {row.date:%d/%m/%Y}"
    )
    fig = px.line(
        hist, x="date", y="risk_score",
        labels={"date": "วันที่", "risk_score": "คะแนนสาธิต"},
    )
    fig.update_traces(line_color="#16877e")
    fig.update_yaxes(range=[0, 100])

    return (
        text,
        *[indicator_table(row, cols) for _, cols in GROUPS],
        indicator_table(row, ENV, True),
        styled(fig, f"แนวโน้มรายวัน — {site}"),
    )


# ─────────────────────── Conditional 30-day scenarios ───────────────────────

def make_future(site, scenario):
    hist = DATA[DATA.site_id == site].sort_values("date")
    future_dates = pd.date_range(
        hist.date.max() + pd.Timedelta(days=1), periods=30
    )

    # สุ่มบล็อก 30 วันที่ทราบแล้ว โดยพยายามจับคู่เดือนตามปฏิทิน
    starts = [
        i for i in range(len(hist) - 29)
        if hist.iloc[i].date.month == future_dates[0].month
    ]
    rng = np.random.default_rng(SEED + IDS.index(site))
    start = int(rng.choice(starts if starts else np.arange(len(hist) - 29)))
    source = hist.iloc[start:start + 30]

    # ใช้เฉพาะ predictors ไม่มี risk_score จากบล็อกเดิมส่งเข้าโมเดล
    future = source[["site_id"] + EDIT].copy().reset_index(drop=True)
    future["date"] = future_dates

    if scenario == SCENARIOS[1]:
        future.loc[:6, "Rainfall"] = 0
        future.loc[7, "Rainfall"] = 30

    combined = pd.concat([
        hist[["date", "site_id"] + EDIT], future
    ], ignore_index=True)

    return features(combined).tail(30).copy(), source


def forecast_view(site, model_name, scenario):
    hist = DATA[DATA.site_id == site].sort_values("date")
    future, source = make_future(site, scenario)
    future["prediction"] = predict(model_name, future)
    recent = hist.tail(90)

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    rain = pd.concat([
        recent[["date", "Rainfall"]],
        future[["date", "Rainfall"]],
    ])
    fig.add_trace(
        go.Bar(
            x=rain.date, y=rain.Rainfall,
            name="Rainfall (mm)", marker_color="#91c7d6", opacity=0.30,
        ),
        secondary_y=True,
    )
    fig.add_trace(
        go.Scatter(
            x=recent.date, y=recent.risk_score,
            name="คะแนนย้อนหลังจำลอง", line=dict(color="#16877e"),
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=future.date, y=future.prediction,
            name="คาดการณ์ตามสมมติฐาน",
            line=dict(color="#bc5651", dash="dash"),
        ),
        secondary_y=False,
    )

    events = future[future.first_rain == 1]
    fig.add_trace(
        go.Scatter(
            x=events.date, y=events.prediction, mode="markers",
            marker=dict(color="#c29539", size=10, symbol="diamond"),
            name="ฝนแรกหลังแล้ง ≥5 วัน",
        ),
        secondary_y=False,
    )
    fig.add_vline(
        x=hist.date.max().timestamp() * 1000,
        line_dash="dot", line_color="#64748b",
    )
    fig.update_yaxes(
        title_text="คะแนนสาธิต", range=[0, 100], secondary_y=False
    )
    fig.update_yaxes(
        title_text="Rainfall (mm)", rangemode="tozero", secondary_y=True
    )

    table = pd.DataFrame({
        "วันที่": future.date.dt.strftime("%Y-%m-%d"),
        "ฝนสมมติ (mm)": future.Rainfall.round(1),
        "ฝน lag 1 วัน": future.rain_lag1.round(1),
        "ฝน lag 3 วัน": future.rain_lag3.round(1),
        "ฝนสะสม 7 วันก่อนหน้า": future.rain_prev7.round(1),
        "วันแห้งก่อนหน้า": future.dry_before.astype(int),
        "ฝนแรกหลังแล้ง": future.first_rain.map({0: "ไม่ใช่", 1: "ใช่"}),
        "คะแนนคาดการณ์": future.prediction.round(1),
        "ระดับ": future.prediction.map(category),
    })

    assumption = (
        "กรณีทดสอบนี้ตั้งฝน 7 วันแรกเป็น 0 และวันที่ 8 เป็น 30 mm; "
        "ตัวแปรอื่นคงตามบล็อกสมมติ"
        if scenario == SCENARIOS[1]
        else "กรณีนี้ใช้ค่าฝนและตัวแปรอื่นตามบล็อกสมมติ "
             "โดยไม่บังคับเหตุ first-rain"
    )
    note = (
        f"**สถานการณ์สมมติ ไม่ใช่พยากรณ์อากาศ:** "
        f"ใช้บล็อกย้อนหลังของจุดเดียวกัน "
        f"({source.date.min():%d/%m/%Y}–{source.date.max():%d/%m/%Y}) "
        f"เป็นค่าอนาคต 30 วัน รวม environmental/MST/eDNA/pathogen inputs "
        f"ไม่ได้ใช้ผลตรวจในอนาคต  \n"
        f"{assumption} จึงเป็น conditional scenario "
        f"ไม่ใช่แบบจำลองการแพร่เชื้อ  \n"
        f"lag ใช้เฉพาะฝนก่อนวันนั้น · ช่วงอนาคตพบ first-rain "
        f"{len(events)} วัน โมเดลเรียนรู้ความสัมพันธ์จากข้อมูล "
        f"ไม่กำหนดให้ฝนเพิ่มความเสี่ยงเสมอ"
    )
    return (
        styled(fig, f"{site} · {model_name} · 30 วัน", 450),
        table,
        note,
    )


# ─────────────────────── Rule-based community tasks ───────────────────────

ROLE_TASKS = {
    "ผู้บริหารนโยบาย": [
        "กำหนดเจ้าของงานและตรวจช่องว่างข้อมูล",
        "ทบทวนทรัพยากรสำหรับการตรวจยืนยัน",
        "ประสานหน่วยงานตามขอบเขตหน้าที่",
        "กำหนดวันติดตามและหลักฐานที่ต้องทบทวน",
    ],
    "นักท่องเที่ยว": [
        "ตรวจประกาศพื้นที่ก่อนทำกิจกรรม",
        "บันทึกสิ่งผิดปกติจากพื้นที่ที่เข้าถึงได้อย่างปลอดภัย",
        "ส่งข้อมูลให้ผู้ดูแลพื้นที่โดยไม่ระบุข้อมูลส่วนบุคคล",
        "ติดตามประกาศและคำแนะนำของพื้นที่",
    ],
    "เจ้าหน้าที่ดูแลแหล่งน้ำ": [
        "ตรวจคุณภาพข้อมูลและสถานะเครื่องมือ",
        "เสนอแผนตรวจยืนยันตาม SOP ที่อนุมัติ",
        "ประสานการจัดการตามแผนของหน่วยงาน",
        "บันทึกผลติดตามก่อนเสนอเปลี่ยนมาตรการ",
    ],
    "ประชาชนในพื้นที่": [
        "บันทึกตำแหน่งและเวลาที่พบปัญหา",
        "ระบุกิจกรรมใช้น้ำและการเปลี่ยนแปลงที่สังเกต",
        "ส่งรายงานให้ผู้ประสานงานพื้นที่",
        "ติดตามผลยืนยันและประกาศของพื้นที่",
    ],
    "ผู้นำชุมชน": [
        "รวมรายงานที่เกี่ยวข้องกับเหตุเดียวกัน",
        "กำหนดผู้ประสานงานและข้อมูลที่ยังขาด",
        "นัดหมายการติดตามร่วมกับผู้รับผิดชอบ",
        "สื่อสารสถานะโดยแยกสิ่งที่ทราบกับสิ่งที่ยังไม่ยืนยัน",
    ],
    "เกษตรกรผู้เลี้ยงปลา": [
        "ตรวจระบบน้ำตาม SOP ของฟาร์มที่เกี่ยวข้อง",
        "รวบรวมประวัติระบบน้ำและสุขภาพสัตว์น้ำ",
        "ส่งข้อมูลให้ผู้เชี่ยวชาญประเมินโดยไม่เลือกยาเอง",
        "บันทึกการดำเนินงานและผลติดตามตามแผน",
    ],
    "ชาวประมง": [
        "ตรวจประกาศที่เกี่ยวข้องกับกิจกรรมประมง",
        "บันทึกความผิดปกติระหว่างกิจกรรมปกติ",
        "ส่งข้อมูลพื้นที่ให้ผู้รับผิดชอบทบทวน",
        "ติดตามผลตรวจ โดยไม่ใช้คะแนนนี้เลือกทะเลที่ไกลหรือลึกกว่า",
    ],
}
ROLES = list(ROLE_TASKS)
ACTION_NOTE = (
    "ข้อเสนอแนะเหล่านี้เป็นแนวทางเบื้องต้น\n"
    "ซึ่งสามารถพัฒนาร่วมกับชุมชนเพื่อให้เหมาะสม\n"
    "กับบริบทพื้นที่และระดับความเสี่ยงในอนาคต"
)


def task_context(site, role):
    row = LATEST.loc[site]
    relevant = (
        f"Rainfall {row.Rainfall:.1f} mm; "
        f"DO {row.DO:.1f} mg/L; pH {row.pH:.2f}"
    )
    flags = []
    if row.DO < 4:
        flags.append("ทบทวนค่า DO และระบบตรวจวัดตาม SOP")
    if row.Rainfall >= 20 or row.first_rain:
        flags.append("ตรวจข้อมูลฝนก่อนหน้าและผลหน่วงเวลา")
    if not 7 <= row.pH <= 8.5:
        flags.append("ตรวจค่า pH ซ้ำและที่มาของข้อมูล")
    if row.Temperature > 32:
        flags.append("ทบทวนข้อมูลอุณหภูมิและบริบทการใช้น้ำ")

    tasks = list(ROLE_TASKS[role])
    tasks[0] += f" — {category(row.risk_score)}: {action(row.risk_score)}"
    tasks[1] += (
        " — "
        + ("; ".join(flags) if flags else "ติดตามตัวแปรตามแผนพื้นที่")
        + f" ({relevant}; เกณฑ์คัดงานสมมติ)"
    )
    return dict(
        site=site, role=role, score=float(row.risk_score),
        category=category(row.risk_score), tasks=tasks,
    )


def task_headers(ctx):
    return [
        f"### ภารกิจ {i+1}\n{task}"
        for i, task in enumerate(ctx["tasks"])
    ]


def task_summary(ctx):
    return (
        f"**{ctx['site']} · {ctx['role']} · "
        f"คะแนนปัจจุบันจำลอง {ctx['score']:.1f}/100 · {ctx['category']}**\n"
        "งานอ้างอิงคะแนนปัจจุบันจากสูตรสาธิต ไม่ใช่ผลคาดการณ์อนาคต"
    )


def reset_tasks(site, role):
    ctx = task_context(site, role)
    return (
        ctx, task_summary(ctx), *task_headers(ctx),
        *([""] * 4), *([None] * 4), "", None,
    )


def save_tasks(ctx, site, role, *payload):
    if not ctx or ctx["site"] != site or ctx["role"] != role:
        raise gr.Error(
            "พื้นที่หรือบทบาทเพิ่งเปลี่ยน โปรดรอให้ภารกิจอัปเดตก่อนส่ง"
        )

    texts, images = payload[:4], payload[4:]
    if any(
        not isinstance(t, str) or not t.strip() or len(t) > 2000
        for t in texts
    ):
        raise gr.Error(
            "กรอกคำตอบทั้ง 4 ข้อ ข้อละไม่เกิน 2,000 ตัวอักษร; "
            "รูปเป็นตัวเลือก"
        )

    submission = uuid.uuid4().hex
    folder = OUT / submission
    folder.mkdir()
    rows = []

    for i, (text, image) in enumerate(zip(texts, images)):
        image_name = ""
        if image:
            source = Path(image)
            if not source.is_file() or source.stat().st_size > 10 * 1024 * 1024:
                raise gr.Error(
                    "รูปต้องเป็นไฟล์ที่อัปโหลดและมีขนาดไม่เกิน 10 MB"
                )
            destination = folder / f"task_{i+1}{source.suffix.lower()}"
            shutil.copy2(source, destination)
            image_name = str(destination.relative_to(OUT))

        rows.append(dict(
            submission_id=submission,
            timestamp_utc=datetime.now(timezone.utc).isoformat(),
            site_id=site,
            role=role,
            current_demo_risk=ctx["score"],
            risk_category=ctx["category"],
            task_number=i+1,
            task=ctx["tasks"][i],
            response=text.strip(),
            image=image_name,
            status="submitted_unreviewed",
            data_status="SYNTHETIC_DEMO",
        ))

    # ป้องกันข้อความกลายเป็นสูตรเมื่อเปิด CSV ใน spreadsheet
    for row in rows:
        for key, value in row.items():
            if isinstance(value, str) and value.lstrip().startswith(
                ("=", "+", "-", "@")
            ):
                row[key] = "'" + value

    receipt = folder / "receipt.csv"
    with SAVE_LOCK:
        pd.DataFrame(rows).to_csv(
            receipt, index=False, encoding="utf-8-sig"
        )
        aggregate = OUT / "submissions.csv"
        with aggregate.open("a", newline="", encoding="utf-8-sig") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
            if stream.tell() == 0:
                writer.writeheader()
            writer.writerows(rows)

    return (
        f"✅ บันทึก 4 ภารกิจแล้ว · รหัส {submission[:10]} · "
        "รอตรวจทาน ยังไม่ยืนยันคุณภาพน้ำ",
        str(receipt),
    )


# ─────────────────────── Trained-model simulator ───────────────────────

def simulate(site, model_name, *values):
    if site not in IDS or model_name not in MODELS or len(values) != len(EDIT):
        raise gr.Error("โปรดเลือกจุด โมเดล และกรอก input ให้ครบ")

    original = LATEST.loc[site]
    row = original.copy()

    for name, value in zip(EDIT, values):
        lo, hi = RANGES.get(name, (0, 100))
        if value is None or not np.isfinite(value) or not lo <= float(value) <= hi:
            raise gr.Error(
                f"{name}: โปรดระบุตัวเลขระหว่าง {lo}–{hi}"
            )
        row[name] = float(value)

    row["site_id"] = site
    row["first_rain"] = int(
        row.Rainfall >= 10 and row.dry_before >= 5
    )
    base_row = original.copy()
    base_row["site_id"] = site

    new = float(predict(model_name, pd.DataFrame([row]))[0])
    baseline = float(predict(model_name, pd.DataFrame([base_row]))[0])

    # Sensitivity: แทนทีละตัวด้วยค่าเฉลี่ย TRAIN
    alternatives = []
    for name in EDIT:
        alternate = row.copy()
        alternate[name] = float(TRAIN_MEANS[name])
        alternate["first_rain"] = int(
            alternate.Rainfall >= 10 and alternate.dry_before >= 5
        )
        alternatives.append(alternate)

    deltas = new - predict(model_name, pd.DataFrame(alternatives))
    ranking = pd.DataFrame({"ตัวแปร": EDIT, "ผลต่างคะแนน": deltas})
    ranking = ranking.iloc[
        np.argsort(abs(ranking["ผลต่างคะแนน"].to_numpy()))[-8:]
    ]

    fig = go.Figure(go.Bar(
        x=ranking["ผลต่างคะแนน"],
        y=ranking["ตัวแปร"],
        orientation="h",
        marker_color=[
            "#bc5651" if v >= 0 else "#16877e"
            for v in ranking["ผลต่างคะแนน"]
        ],
    ))
    fig.add_vline(x=0, line_color="#64748b")
    fig.update_xaxes(
        title="Δ score vs training-mean replacement"
    )

    note = (
        f"ค่าทำนายของโมเดลเมื่อไม่แก้ input = **{baseline:.1f}**; "
        f"ผลต่างจากการแก้ input ภายในโมเดลเดียวกัน = "
        f"**{new-baseline:+.1f}** คะแนน  \n"
        "ผลต่างเทียบคะแนนอ้างอิงด้านบนรวมความคลาดเคลื่อนของโมเดลด้วย  \n"
        f"**Action สาธิต:** {action(new)}  \n"
        "**ไม่ใช่เหตุและผล:** sensitivity เปลี่ยนทีละตัวไปค่าเฉลี่ย train; "
        "ตัวแปรที่สัมพันธ์กันอาจให้สถานการณ์ที่ไม่เกิดจริง "
        "ส่วนต่างไม่จำเป็นต้องรวมเท่าคะแนน · "
        "lag ฝนก่อนหน้าคงตามประวัติของจุดนั้น"
    )
    return (
        float(original.risk_score),
        new,
        new - float(original.risk_score),
        category(new),
        styled(fig, "อะไรมีผลต่อคะแนนครั้งนี้?"),
        note,
    )


def load_sim_site(site):
    row = LATEST.loc[site]
    lag_note = (
        f"ฝน lag 1 วัน {row.rain_lag1:.1f} mm · "
        f"lag 3 วัน {row.rain_lag3:.1f} mm · "
        f"ฝนสะสม 7 วันก่อนหน้า {row.rain_prev7:.1f} mm · "
        f"วันแห้งก่อนหน้า {int(row.dry_before)} วัน"
    )
    return (*[float(row[k]) for k in EDIT], lag_note)


# ─────────────────────── Logical consistency checks ───────────────────────

assert TRAIN.date.max() < TEST.date.min()
assert "risk_score" not in XCOLS

sample = DATA[DATA.site_id == IDS[0]].sort_values("date")
assert np.allclose(
    sample.rain_lag1.iloc[1:], sample.Rainfall.iloc[:-1]
)

for site in IDS:
    future_check, _ = make_future(site, SCENARIOS[1])
    assert len(future_check) == 30
    assert int(future_check.iloc[7].first_rain) == 1

for name in MODEL_NAMES:
    checked = predict(name, TEST.head(12))
    assert np.isfinite(checked).all()
    assert ((checked >= 0) & (checked <= 100)).all()




# Optional GIS ingestion. Uploaded layers affect maps only, never ML inputs/targets.
import json
import html
import base64
import io
import math
import os

GIS_DIR = Path('/content/gis') if Path('/content').exists() else Path.cwd() / 'gis'
GIS_FILES = {'catchments': 'catchments_B01_B12.geojson', 'flows': 'flow_paths.geojson',
             'stations': 'monitoring_sites.geojson', 'slope': 'terrain_slope.tif',
             'groundwater': 'groundwater_screening.tif'}
LAYER_NAMES = ['Drainage catchments', 'Flow paths', 'Monitoring points']
RASTER_NAMES = ['None', 'Terrain slope', 'Groundwater screening — UNVALIDATED']


def checked_path(path, limit=100):
    p = Path(path)
    if not p.is_file() or p.stat().st_size > limit * 1024**2:
        raise ValueError(f'File missing or larger than {limit} MB')
    return p


def validate_position(x, y):
    # Deliberately local to Ko Sichang; rejects most projected/unrelated coordinates.
    if not (math.isfinite(x) and math.isfinite(y) and 100.65 <= x <= 101.0 and 13.0 <= y <= 13.30):
        raise ValueError('Expected WGS84 lon,lat near Ko Sichang (100.65–101.0 E, 13.0–13.30 N). Reproject in R/GIS first.')


def coordinates_iter(coords):
    if not isinstance(coords, (list, tuple)) or not coords:
        raise ValueError('Empty or malformed geometry')
    if isinstance(coords[0], (int, float)):
        if len(coords) < 2:
            raise ValueError('A coordinate needs longitude and latitude')
        validate_position(float(coords[0]), float(coords[1]))
        yield coords[:2]
    else:
        for child in coords:
            yield from coordinates_iter(child)


def read_geojson(path, kind):
    from shapely.geometry import shape
    obj = json.loads(checked_path(path, 20).read_text(encoding='utf-8-sig'))
    crs = obj.get('crs')
    if crs and not any(s in str(crs).upper() for s in ['4326', 'CRS84']):
        raise ValueError('Export GeoJSON in EPSG:4326, longitude then latitude')
    feats = obj.get('features', [])
    if obj.get('type') != 'FeatureCollection' or not 1 <= len(feats) <= 20000:
        raise ValueError('Expected a non-empty GeoJSON FeatureCollection, at most 20,000 features')
    allowed = {'catchments': ['Polygon', 'MultiPolygon'], 'flows': ['LineString', 'MultiLineString'], 'stations': ['Point']}[kind]
    ids, ncoords = [], 0
    for f in feats:
        g = f.get('geometry') or {}
        if g.get('type') not in allowed:
            raise ValueError(f'{kind} requires {allowed}')
        ncoords += sum(1 for _ in coordinates_iter(g.get('coordinates')))
        if ncoords > 300000:
            raise ValueError('Simplify this layer to at most 300,000 coordinates before uploading')
        geom = shape(g)
        if geom.is_empty or not geom.is_valid:
            raise ValueError('Invalid geometry; repair in GIS software before export')
        if kind != 'flows':
            sid = f.get('properties', {}).get('site_id')
            if sid not in IDS:
                raise ValueError('Each catchment/station requires properties.site_id B01–B12')
            ids.append(sid)
        # Keep only required attributes. Arbitrary uploaded HTML is not rendered.
        f['properties'] = {'site_id': f.get('properties', {}).get('site_id', '')}
    if len(ids) != len(set(ids)):
        raise ValueError('Duplicate site_id; dissolve catchments to one (Multi)Polygon per B ID')
    if kind == 'catchments' and set(ids) != set(IDS):
        raise ValueError('Catchments must include exactly B01–B12')
    return {'type': 'FeatureCollection', 'features': feats}


def raster_layer(path, kind):
    import rasterio
    from rasterio.warp import calculate_default_transform, reproject, Resampling, transform_bounds
    from PIL import Image
    with rasterio.open(checked_path(path)) as src:
        if src.crs is None or src.count != 1:
            raise ValueError('GeoTIFF needs a defined CRS and exactly one numeric band')
        if src.width * src.height > 100_000_000:
            raise ValueError('Raster too large; crop/resample to Ko Sichang first')
        west, south, east, north = transform_bounds(src.crs, 'EPSG:4326', *src.bounds)
        validate_position(west, south)
        validate_position(east, north)
        if east <= west or north <= south:
            raise ValueError('Invalid raster extent')
        # Reproject to Web Mercator so the image overlay matches the tile map.
        tr, width, height = calculate_default_transform(src.crs, 'EPSG:3857', src.width, src.height, *src.bounds)
        factor = min(1, 600 / max(width, height))
        out_w, out_h = max(1, round(width * factor)), max(1, round(height * factor))
        tr = tr * rasterio.Affine.scale(width / out_w, height / out_h)
        dst = np.full((out_h, out_w), np.nan, dtype=np.float32)
        reproject(source=rasterio.band(src, 1), destination=dst,
                  src_transform=src.transform, src_crs=src.crs, src_nodata=src.nodata,
                  dst_transform=tr, dst_crs='EPSG:3857', dst_nodata=np.nan,
                  resampling=Resampling.nearest)
        left, bottom, right, top = rasterio.transform.array_bounds(out_h, out_w, tr)
        west, south, east, north = transform_bounds('EPSG:3857', 'EPSG:4326', left, bottom, right, top)
    valid = np.isfinite(dst)
    if not valid.any():
        raise ValueError('Raster has no valid cells')
    if kind == 'slope':
        if np.any((dst[valid] < 0) | (dst[valid] > 90)):
            raise ValueError('Slope raster must be degrees, 0–90; set NoData correctly')
        # Fixed 0–45 degree legend; steep values saturate at 45.
        stops = np.array([[232, 244, 197], [95, 184, 125], [21, 105, 85], [126, 84, 39]])
        t = np.nan_to_num(np.clip(dst / 45, 0, 1)) * 3
        low = np.floor(t).astype(int)
        high = np.minimum(low + 1, 3)
        rgb = stops[low] * (1 - (t-low)[..., None]) + stops[high] * (t-low)[..., None]
    else:
        if not np.isin(dst[valid], [1, 2, 3]).all():
            raise ValueError('Screening raster: encode 1=gentle (<5°), 2=moderate (5–15°), 3=steep (>15°). Other values must be NoData.')
        palette = np.array([[0, 0, 0], [35, 139, 69], [161, 217, 155], [244, 216, 171]])
        rgb = palette[np.nan_to_num(dst).astype(int)]
    rgba = np.dstack([rgb.astype(np.uint8), valid.astype(np.uint8) * 200])
    buf = io.BytesIO()
    Image.fromarray(rgba).save(buf, format='PNG')
    return dict(sourcetype='image', source='data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode(),
                coordinates=[[west, north], [east, north], [east, south], [west, south]], opacity=0.80)


def load_gis_bundle(paths):
    bundle, messages = {}, []
    for kind, path in paths.items():
        if not path:
            continue
        try:
            bundle[kind] = read_geojson(path, kind) if kind in ['catchments', 'flows', 'stations'] else raster_layer(path, kind)
            messages.append(f'{kind}: loaded')
        except Exception as exc:
            messages.append(f'{kind}: REJECTED — {exc}')
    bundle['status'] = '; '.join(messages) if messages else 'No GIS files loaded. Demonstration coordinates only.'
    return bundle


INITIAL_GIS = load_gis_bundle({k: GIS_DIR/v for k, v in GIS_FILES.items() if (GIS_DIR/v).is_file()})


def gis_status(bundle, raster='None'):
    text = html.escape((bundle or {}).get('status', 'No GIS files loaded. Demonstration coordinates only.'))
    legend = ''
    if raster == RASTER_NAMES[1]:
        legend = '<br><b>Slope (degrees):</b> pale green 0° → green 15° → dark teal 30° → brown ≥45°.'
    elif raster == RASTER_NAMES[2]:
        legend = '<br><b>Slope-only screening:</b> dark green &lt;5° · light green 5–15° · sand &gt;15°. UNVALIDATED: gentle slopes do not prove groundwater; no aquifer, yield, well or recharge measurements.'
    key = 'slope' if raster == RASTER_NAMES[1] else 'groundwater' if raster == RASTER_NAMES[2] else None
    if key and key not in (bundle or {}):
        legend += '<br><b>Selected raster is unavailable. Upload its GeoTIFF to display it.</b>'
    return f'<div class="ow-banner"><b>GIS / DATA STATUS</b><br>{text}{legend}<br>Uploaded geometry is user-supplied; accuracy and B-ID associations require review. All water-quality values and risk scores remain synthetic.</div>'


def map_plot(bundle=None, layers=None, raster='None', show_risk=False):
    bundle = bundle or {}
    layers = LAYER_NAMES if layers is None else layers
    fig = go.Figure()
    if 'Drainage catchments' in layers and 'catchments' in bundle:
        feats = bundle['catchments']['features']
        ids = [f['properties']['site_id'] for f in feats]
        values = [float(LATEST.loc[s].risk_score) for s in ids] if show_risk else [IDS.index(s) for s in ids]
        fig.add_trace(go.Choroplethmap(
            geojson=bundle['catchments'], locations=ids, featureidkey='properties.site_id', z=values,
            zmin=0, zmax=100 if show_risk else 11,
            colorscale=[[0,'#16877e'],[.5,'#edc873'],[1,'#bf5651']] if show_risk else 'Tealgrn',
            marker_opacity=.48, marker_line_width=1.3, marker_line_color='#366a70',
            showscale=bool(show_risk), colorbar=dict(title='Demo risk', thickness=10, len=.6),
            customdata=ids, name='User-supplied modeled catchments',
            hovertemplate='%{customdata}<br>Modeled catchment / user-supplied' + ('<br>Synthetic score %{z:.1f} / 100 (illustrative B-ID join)' if show_risk else '') + '<extra></extra>'))
    if 'Flow paths' in layers and 'flows' in bundle:
        xs, ys = [], []
        for f in bundle['flows']['features']:
            g = f['geometry']
            for line in ([g['coordinates']] if g['type']=='LineString' else g['coordinates']):
                xs.extend([p[0] for p in line] + [None]); ys.extend([p[1] for p in line] + [None])
        fig.add_trace(go.Scattermap(lon=xs, lat=ys, mode='lines', line=dict(color='#2c91ba',width=2),
                                   name='Modeled flow paths', hoverinfo='name'))
    if 'Monitoring points' in layers:
        if 'stations' in bundle:
            rows = [(f['properties']['site_id'], *f['geometry']['coordinates'][:2]) for f in bundle['stations']['features']]
            d = pd.DataFrame(rows, columns=['site_id','longitude','latitude']).merge(LATEST.reset_index().drop(columns=['longitude','latitude']),on='site_id')
            point_label = 'Supplied coordinates · synthetic values'
        else:
            d = LATEST.reset_index()
            point_label = 'SIMULATED coordinates · synthetic values'
        fig.add_trace(go.Scattermap(lon=d.longitude,lat=d.latitude,mode='markers',
            marker=dict(size=15,color=d.risk_score,cmin=0,cmax=100,colorscale=[[0,'#16877e'],[.5,'#edc873'],[1,'#bf5651']]),
            customdata=np.column_stack([d.site_id,d.risk_score]),name=point_label,
            hovertemplate='%{customdata[0]}<br>'+point_label+'<br>Demo risk %{customdata[1]:.1f}/100<extra></extra>'))
    if not fig.data:
        fig.add_trace(go.Scattermap(lon=[100.811], lat=[13.153], marker=dict(size=0), hoverinfo='skip',showlegend=False))
    overlays = []
    key = 'slope' if raster == RASTER_NAMES[1] else 'groundwater' if raster == RASTER_NAMES[2] else None
    if key in bundle:
        overlays.append(dict(bundle[key], below='traces'))
    fig.update_layout(map=dict(style='carto-positron',center=dict(lat=13.153,lon=100.811),zoom=11.3,layers=overlays),
                      showlegend=False, uirevision='ko-sichang-map')
    return styled(fig, 'Ko Sichang · optional GIS / synthetic monitoring', 490)


def upload_gis(catchments, flows, stations, slope, groundwater, layers, raster, show_risk):
    bundle = load_gis_bundle(dict(zip(GIS_FILES, [catchments,flows,stations,slope,groundwater])))
    return bundle, map_plot(bundle,layers,raster,show_risk), gis_status(bundle,raster)


def refresh_map(bundle, layers, raster, show_risk):
    return map_plot(bundle,layers,raster,show_risk), gis_status(bundle,raster)


# Shared presentation helpers; numerical/model functions above retain original behavior.
def kpi(label, value, foot='SYNTHETIC DATA'):
    return f'<div class="ow-kpi"><div class="ow-kpi-label">{html.escape(label)}</div><div class="ow-kpi-value">{html.escape(str(value))}</div><div class="ow-kpi-foot">{html.escape(foot)}</div></div>'


def section(title, subtitle='', badge=''):
    return f'<div class="ow-section"><div><h2>{html.escape(title)}</h2><p>{html.escape(subtitle)}</p></div><span class="ow-badge">{html.escape(badge)}</span></div>'


def result_card(site, model, original, score, delta):
    color = '#16877e' if score < 35 else '#c29539' if score < 65 else '#bc5651'
    return f'''<div class="ow-section"><div><h2>ผลประเมินจากโมเดลที่ฝึกแล้ว</h2><p>{html.escape(model)}</p></div><span class="ow-badge">{site}</span></div>
    <div class="ow-score"><div class="ow-ring" role="img" aria-label="Synthetic risk score {score:.1f} out of 100" style="background:conic-gradient({color} {score}%,#ecf1f0 {score}%);"><div class="ow-ring-center"><div class="ow-ring-score" style="color:{color}">{score:.1f}</div><div class="ow-ring-label">คะแนน / 100 · สาธิต</div></div></div>
    <div class="ow-score-text"><h3 style="color:{color}">{category(score)}</h3><p>คะแนนลำดับตรวจสอบสาธิต<br>ไม่ใช่โอกาสเกิดโรคจริง<br>ต่ำ &lt;35 · ปานกลาง 35–&lt;65 · สูง ≥65</p></div></div>
    <div class="ow-action"><strong>{action(score)}</strong>ตรวจความครบถ้วนของข้อมูลและแหล่งที่มา<br>ส่งข้อมูลให้ผู้รับผิดชอบพื้นที่กำหนดการติดตามที่เหมาะสม</div>
    <div class="ow-mini-grid"><div><small>คะแนนอ้างอิง (สูตรจำลอง)</small><b>{original:.1f}</b></div><div><small>ผลต่างเทียบคะแนนอ้างอิง</small><b>{delta:+.1f}</b></div><div><small>สถานะข้อมูล</small><b style="font-size:13px">SYNTHETIC ONLY</b></div></div>'''


def simulator_view(site, model, *values):
    original, score, delta, cat, fig, note = simulate(site,model,*values)
    return result_card(site,model,original,score,delta), fig, note


def reset_simulator(site, model):
    vals = load_sim_site(site)
    return (*vals, *simulator_assess(site,model,*vals[:-1]))


def input_fingerprint(site, model, values):
    return [site, model, *[round(float(v), 2) if v is not None else None for v in values]]


def simulator_assess(site, model, *values):
    return (*simulator_view(site, model, *values), input_fingerprint(site, model, values))


def pending_result(evaluated, site, model, *values):
    if evaluated == input_fingerprint(site, model, values):
        return gr.skip()
    return '<div class="ow-banner"><b>Inputs changed / ข้อมูลเปลี่ยนแล้ว</b><br>กด “ประเมินความเสี่ยง” เพื่ออัปเดตผลด้วยค่าที่แก้ไข</div>'


def performance_plot():
    fig = go.Figure()
    for metric, color in [('MAE','#16877e'),('RMSE','#648391')]:
        fig.add_trace(go.Bar(x=['Ridge','MLP'], y=METRICS[metric],name=metric,marker_color=color))
    fig.update_layout(barmode='group',legend=dict(x=0,y=1.15))
    fig.update_yaxes(title='Score error (lower is better)',rangemode='tozero')
    return styled(fig,'Held-out daily prediction · synthetic data',290)


CSS = ":root {color-scheme:light!important}\nbody,.gradio-container {background:#f4f7f7!important;color:#193b48!important}\n.gradio-container {max-width:1480px!important;margin:auto!important;padding:0 28px 36px!important;font-family:'Noto Sans Thai',Tahoma,Arial,sans-serif!important}\n.ow-header {background:#152f3a;color:white;margin:0 -28px 28px;padding:25px 34px;display:flex;justify-content:space-between;align-items:center;gap:20px}\n.ow-brand {font-size:31px;font-weight:800;letter-spacing:-1px;line-height:1.2}.ow-brand span{color:#87cdbc}.ow-header small{display:block;color:#b2cbd0;letter-spacing:2.2px;font-size:10px;margin-top:8px}.ow-header .ow-badge{border:1px solid #6a9999;color:#d6ebe6;background:#284c54}\n.ow-badge{display:inline-block;background:#eaf5f1;color:#16877e;border-radius:7px;padding:9px 13px;font-size:11px;letter-spacing:.5px}\n.ow-eyebrow{color:#16877e;font-size:11px;font-weight:800;letter-spacing:2px;margin:4px 0 9px}.ow-title{font-size:29px;font-weight:800;line-height:1.4;margin:0;color:#193b48}.ow-subtitle{font-size:14px;color:#6b8590;margin:9px 0 22px;line-height:1.7}\n.ow-banner {background:#fff8e8;border:1px solid #efd99d;color:#88672c;padding:14px 18px;border-radius:10px;font-size:12px;line-height:1.8;margin:2px 0 18px}\n.ow-panel {background:white!important;border:1px solid #d7e4e5!important;border-radius:13px!important;padding:20px!important;box-shadow:0 2px 4px #17374303!important;gap:15px!important;overflow:hidden}\n.ow-kpi{background:#fff;border:1px solid #d7e4e5;border-radius:12px;padding:19px 22px;min-height:112px}.ow-kpi-label{color:#6d858f;font-size:12px}.ow-kpi-value{font-size:30px;font-weight:800;color:#193b48;margin:6px 0}.ow-kpi-foot{font-size:11px;color:#16877e}\n.ow-section {display:flex;align-items:center;justify-content:space-between;gap:12px;border-bottom:1px solid #e7eeee;padding-bottom:15px;margin-bottom:2px}.ow-section h2{font-size:19px;margin:0;font-weight:800;color:#193b48}.ow-section p{font-size:12px;color:#6b8590;margin:6px 0 0}\n.ow-flow {display:grid;grid-template-columns:repeat(5,1fr);gap:12px;margin:5px 0 22px}.ow-flow>div{background:white;border:1px solid #d7e4e5;border-radius:10px;padding:14px 17px}.ow-flow small{display:block;color:#6b8590;font-size:10px;margin-bottom:8px}.ow-flow strong{font-size:13px;color:#193b48}\n.ow-score{display:flex;align-items:center;gap:25px;flex-wrap:wrap;margin:6px 0 20px}.ow-ring{width:170px;height:170px;border-radius:50%;display:flex;align-items:center;justify-content:center;flex:none}.ow-ring-center{width:147px;height:147px;border-radius:50%;background:white;display:flex;align-items:center;justify-content:center;flex-direction:column}.ow-ring-score{font-size:43px;font-weight:800;line-height:1.3}.ow-ring-label{font-size:11px;color:#6b8590}.ow-score-text h3{margin:0 0 10px;font-size:24px}.ow-score-text p{color:#6b8590;font-size:12px;line-height:1.8;margin:0}.ow-action{padding:19px;background:#eff7f4;border:1px solid #d2e8df;border-radius:10px;color:#193b48;font-size:13px;line-height:1.8}.ow-action strong{display:block;margin-bottom:7px}\n.ow-mini-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin:14px 0}.ow-mini-grid>div{padding:12px;background:#f6f9f9;border-radius:8px}.ow-mini-grid small{display:block;color:#6b8590;font-size:10px}.ow-mini-grid b{font-size:21px;color:#193b48}\n.ow-footer{margin-top:25px;border-top:1px solid #d7e4e5;padding-top:18px;color:#6b8590;font-size:11px;display:flex;justify-content:space-between;gap:15px}\n#navigation>.tab-nav{background:white;border:1px solid #d7e4e5!important;border-radius:10px;padding:6px!important;margin-bottom:23px;gap:7px}\n#navigation>.tab-nav button{padding:11px 20px!important;font-weight:700!important;color:#6b8590!important;border:0!important;border-radius:7px!important;font-size:13px!important}\n#navigation>.tab-nav button.selected{background:#e8f3f0!important;color:#127d72!important}\n.tabitem{padding:0!important;border:0!important;background:transparent!important;gap:20px!important}\n.gradio-container button.primary{background:#16877e!important;border:1px solid #16877e!important;color:white!important;border-radius:8px!important;font-weight:700!important;min-height:45px}\n.gradio-container .prose h3{color:#193b48!important;font-size:15px!important}.gradio-container .prose p,.gradio-container .prose li{line-height:1.8!important;font-size:12px}\n.gradio-container input,.gradio-container textarea{background:#f9fbfb!important;color:#193b48!important}.gradio-container label span{font-size:12px!important}.gradio-container .plot-container{border-radius:10px;overflow:hidden}\n.ow-task{background:#f9fbfb!important;border:1px solid #e0e9e9!important;border-radius:10px!important;padding:15px!important}\n@media(max-width:800px){.gradio-container{padding:0 14px 25px!important}.ow-header{margin:0 -14px 20px;padding:20px}.ow-header>.ow-badge{display:none}.ow-title{font-size:23px}.ow-flow{grid-template-columns:repeat(2,1fr)}.ow-flow>div:last-child{grid-column:1/-1}.ow-panel{padding:14px!important}.ow-score{gap:16px}.ow-ring{width:145px;height:145px}.ow-ring-center{width:125px;height:125px}#navigation>.tab-nav button{padding:10px!important;font-size:12px!important}.ow-mini-grid{grid-template-columns:1fr}.ow-kpi{padding:15px;min-height:100px}.ow-footer{flex-direction:column}}\n/* Explicit sizing prevents Gradio's intrinsic plot widths from widening mobile pages. */\n.gradio-container{width:100%!important;min-width:0!important;box-sizing:border-box!important}\n.gradio-container .app{padding:0!important;width:100%!important;min-width:0!important;box-sizing:border-box!important}\n.gradio-container .contain,.gradio-container .wrap,.gradio-container .tabs,.gradio-container .tab-wrapper,.gradio-container .tab-container,.gradio-container .tabitem{min-width:0!important;max-width:100%!important}\n.ow-header .ow-brand{color:#fff!important}.ow-header .ow-brand span{color:#87cdbc!important}\n#navigation .tab-container{display:flex;flex-wrap:wrap;background:white;border:1px solid #d7e4e5;border-radius:10px;padding:6px;margin-bottom:23px;gap:5px}\n#navigation .tab-container button{padding:11px 17px!important;border:0!important;border-radius:7px!important;color:#6b8590!important;font-size:13px!important;font-weight:700!important}\n#navigation .tab-container button.selected{background:#e8f3f0!important;color:#127d72!important}\n.gradio-container .html-container{padding:0!important}\n.gradio-container{--block-label-background-fill:transparent!important;--block-label-text-color:#54717c!important}\n.gradio-container .js-plotly-plot,.gradio-container .plot-container,.gradio-container .plotly{width:100%!important;max-width:100%!important;min-width:0!important}\n@media(max-width:800px){.gradio-container .row{flex-direction:column!important;flex-wrap:wrap!important}.gradio-container .row>.column,.gradio-container .row>.block,.gradio-container .row>.form{width:100%!important;min-width:0!important;flex-basis:auto!important;box-sizing:border-box!important}.gradio-container .column{min-width:0!important}.ow-section{flex-wrap:wrap}.ow-section h2{font-size:17px}#navigation .tab-container{gap:3px}#navigation .tab-container button{width:100%;text-align:left;font-size:12px!important;padding:10px!important}.ow-eyebrow{letter-spacing:1px}.ow-header{max-width:none!important}.ow-mini-grid{grid-template-columns:1fr 1fr}.ow-mini-grid>div:last-child{grid-column:1/-1}}\n\n#navigation .tab-wrapper,#navigation .tab-container{height:auto!important;overflow:visible!important;max-height:none!important;min-height:44px!important}#navigation .tab-container:after{display:none!important}\n"
THEME = gr.themes.Soft(primary_hue='teal',neutral_hue='slate',font=['Noto Sans Thai','Tahoma','Arial','sans-serif']).set(
    body_background_fill='#f4f7f7',body_background_fill_dark='#f4f7f7',
    body_text_color='#193b48',body_text_color_dark='#193b48',
    block_background_fill='white',block_background_fill_dark='white',
    block_border_color='#d7e4e5',block_border_color_dark='#d7e4e5',
    input_background_fill='#f9fbfb',input_background_fill_dark='#f9fbfb',
    input_border_color='#cfdee0',input_border_color_dark='#cfdee0',
    button_primary_background_fill='#16877e',button_primary_background_fill_dark='#16877e',
    button_primary_text_color='white',button_primary_text_color_dark='white',
)

if 'demo' in globals():
    demo.close()
home0 = home_view(DEFAULT)
forecast0 = forecast_view(DEFAULT,MODEL_NAMES[0],SCENARIOS[0])
context0 = task_context(DEFAULT,ROLES[0])
sim0 = simulator_view(DEFAULT,MODEL_NAMES[0],*[float(LATEST.loc[DEFAULT,k]) for k in EDIT])

with gr.Blocks(title='OneWaterAI | Ko Sichang',theme=THEME,css=CSS,analytics_enabled=False) as demo:
    gr.HTML('''<header class="ow-header"><div><div class="ow-brand">OneWater<span>AI</span></div><small>ENVIRONMENTAL HEALTH · WORKABLE PROTOTYPE</small></div><span class="ow-badge">KO SICHANG / เกาะสีชัง</span></header>
    <div class="ow-eyebrow">MULTI-MARKER × ENVIRONMENT × MACHINE LEARNING</div><h1 class="ow-title">จากข้อมูล สู่คะแนนและงานที่ทำต่อได้</h1><p class="ow-subtitle">สำรวจพื้นที่ · ทดสอบสถานการณ์ · ติดตามงานร่วมกับชุมชน</p>
    <div class="ow-banner"><b>ข้อมูลจำลองเพื่อการสาธิตเท่านั้น / SYNTHETIC MODEL EXPERIMENT</b><br>ตัวแปรและฉลากสร้างขึ้นเพื่อทดสอบระบบ ไม่ใช่ผลตรวจหรือความแม่นยำภาคสนามของเกาะสีชัง ไม่ยืนยันความปลอดภัยของน้ำหรืออาหารทะเล · ลิงก์สาธารณะชั่วคราว: ไม่ใส่ข้อมูลส่วนบุคคล · ข้อมูลและรูปเก็บชั่วคราวใน Colab</div>''')
    with gr.Tabs(elem_id='navigation'):
        with gr.Tab('01  Dashboard · ภาพรวม',id='home'):
            with gr.Row():
                gr.HTML(kpi('จุดติดตามสาธิต',12,'B01–B12 · พิกัดจำลอง'))
                gr.HTML(kpi('คะแนนเฉลี่ยล่าสุด',f'{LATEST.risk_score.mean():.1f}','DEMO SCORE / 100'))
                gr.HTML(kpi('จุดระดับสูง · สาธิต',int((LATEST.risk_score>=65).sum()),'เกณฑ์สาธิต ≥65'))
                gr.HTML(kpi('ข้อมูลจำลองล่าสุด',f'{LATEST.date.max():%d %b %Y}','450 DAYS · 12 SITES'))
            with gr.Row(equal_height=False):
                with gr.Column(scale=7,min_width=320,elem_classes=['ow-panel']):
                    gr.HTML(section('Coastal monitoring map','เลือกชั้นข้อมูลและดูรายละเอียดเมื่อชี้จุด','KO SICHANG'))
                    gis_state = gr.State(INITIAL_GIS)
                    map_component = gr.Plot(map_plot(INITIAL_GIS),show_label=False)
                    layer_choice = gr.CheckboxGroup(LAYER_NAMES,value=LAYER_NAMES,label='Map layers')
                    with gr.Row():
                        raster_choice = gr.Dropdown(RASTER_NAMES,value='None',label='Raster overlay')
                        risk_join = gr.Checkbox(False,label='Illustrative B-ID join: synthetic risk on catchments')
                    gis_note = gr.HTML(gis_status(INITIAL_GIS))
                with gr.Column(scale=4,min_width=300,elem_classes=['ow-panel']):
                    gr.HTML(section('Site overview','ค่าทั้งหมดเป็นข้อมูลจำลอง','LATEST'))
                    hsite = gr.Dropdown(IDS,value=DEFAULT,label='เลือกจุดติดตาม')
                    detail = gr.Markdown(home0[0])
                    env_table = gr.Dataframe(home0[5],interactive=False,label='Environmental · ค่าจำลอง')
                    gr.Markdown('พิกัดบนแผนที่เป็นพิกัดจำลองจนกว่าจะอัปโหลด monitoring_sites.geojson\n\nเลือกจุดจากรายการนี้ได้แม้พื้นแผนที่โหลดไม่สำเร็จ')
            with gr.Column(elem_classes=['ow-panel']):
                gr.HTML(section('Historical signals','ประวัติรายวันของจุดที่เลือก · ไม่ใช่ข้อมูลภาคสนาม','450 DAYS'))
                historical = gr.Plot(home0[6],show_label=False)
                tables=[]
                for i,(title,cols) in enumerate(GROUPS):
                    with gr.Accordion(title,open=False):
                        tables.append(gr.Dataframe(home0[i+1],interactive=False))
            hsite.change(home_view,hsite,[detail,*tables,env_table,historical],api_name='site_overview')
            with gr.Accordion('Optional GIS · อัปโหลดชั้นข้อมูลจาก R / GIS',open=False):
                gr.Markdown('''**Original R/GIS files are still needed.** The screenshot is a reference figure only; it is never digitized or used as coordinate data.

Upload any subset below and press **Load GIS files**. Each click replaces this browser session’s GIS bundle. To preload for everyone, place files with the documented names in `/content/gis/` before running the application cell. Reload the app after adding preloaded files.

GeoJSON must be **EPSG:4326, longitude–latitude**, near Ko Sichang. Catchments: exactly one Polygon/MultiPolygon per `site_id` B01–B12. Flow paths: LineString/MultiLineString. Optional monitoring points: Point with `site_id` (a subset is allowed). Catchment labels do not establish a verified station association. Risk coloring is an explicit illustrative B-ID join.

Slope: single-band georeferenced GeoTIFF in **degrees (0–90)**. Groundwater screening: single-band GeoTIFF with **1=&lt;5°, 2=5–15°, 3=&gt;15°**, other cells NoData. Defined CRS and NoData required. CRS reprojection is automatic for rasters. GeoJSON limit 20 MB / 300,000 vertices; GeoTIFF limit 100 MB. Rasters display at up to 600 pixels and are visualization layers only.

**Groundwater is an unvalidated slope-only screening proxy.** No measured groundwater availability, aquifer extent, depth, well yield or recharge is inferred. Modeled drainage pathways are not surveyed rivers. User-supplied layers still require provenance and accuracy review.''')
                with gr.Row():
                    catch_file=gr.File(label='catchments_B01_B12.geojson',file_types=['.geojson','.json'],type='filepath')
                    flow_file=gr.File(label='flow_paths.geojson',file_types=['.geojson','.json'],type='filepath')
                    station_file=gr.File(label='monitoring_sites.geojson (optional)',file_types=['.geojson','.json'],type='filepath')
                with gr.Row():
                    slope_file=gr.File(label='terrain_slope.tif (degrees)',file_types=['.tif','.tiff'],type='filepath')
                    groundwater_file=gr.File(label='groundwater_screening.tif (classes 1–3)',file_types=['.tif','.tiff'],type='filepath')
                load_gis_button=gr.Button('Load GIS files · โหลดชั้นข้อมูล',variant='primary')
                load_gis_button.click(upload_gis,[catch_file,flow_file,station_file,slope_file,groundwater_file,layer_choice,raster_choice,risk_join],[gis_state,map_component,gis_note],api_name=False)
            for control in [layer_choice,raster_choice,risk_join]:
                control.change(refresh_map,[gis_state,layer_choice,raster_choice,risk_join],[map_component,gis_note],api_name=False)
            with gr.Accordion('Scientific method & limitations · สูตรและข้อจำกัด',open=False):
                gr.Markdown('\n**Input:** ชื่อ indicator เป็นคนละกลุ่มและมีหน้าที่ต่างกัน ค่าชีวภาพทั้งหมดเป็นดัชนีจำลอง 0–100 ไม่ใช่ copies/L, read counts, ผลบวก หรือความสามารถก่อโรค\n\n**สูตรอ้างอิงสาธิต:**\n\n`score = clip(100 × [0.03 + 0.20H + 0.08A + 0.26PH + 0.15PA + 0.16D + 0.04|pH−8|/1.5 + 0.02max(T−30,0)/5 + R] + noise, 0, 100)`\n\n- H/A = ค่าเฉลี่ยกลุ่ม human/animal หาร 100\n- PH/PA = ค่าเฉลี่ยเป้าหมายเชื้อมนุษย์/สัตว์น้ำหาร 100\n- D = clip((6−DO)/5, 0, 1)\n- R = 0.08×first_rain×tanh(rain/30) + 0.10×tanh(antecedent7/60)×(1−DO/10) − 0.10×tanh(rain/40)×DO/10\n- noise ~ N(0, 2.2)\n\nน้ำหนักและเกณฑ์ทั้งหมดแต่งขึ้น ไม่ใช่มาตรฐานสุขภาพ ฝนมีทั้งพจน์บวกและลบตามบริบท ไม่บังคับว่าฝนต้องเพิ่มคะแนนเสมอ Biodiversity/eDNA ไม่ใช่เชื้อก่อโรค และไม่มีน้ำหนักเพิ่มความเสี่ยงโดยตรงในสูตรนี้\n\n**โมเดล:** Ridge และ MLPRegressor ใช้ predictors เดียวกัน รวม site ID และ rain lag; scaler fit เฉพาะ train; แยก train/test ตามวันที่ของทุกจุด ใช้โมเดลที่ฝึกชุดเดียวกันกับการคาดการณ์และ simulator\n\n**ระดับสาธิต:** ต่ำ <35, ปานกลาง 35–<65, สูง ≥65 ไม่มีระดับใดยืนยันความปลอดภัย\n\n**ข้อจำกัด:** ชื่อ C16S/qC160F เป็น candidate assay labels ตามข้อเสนอ ต้องตรวจคำจำกัดความและความจำเพาะจากวรรณกรรม; MST ไม่ยืนยันเชื้อก่อโรค; eDNA ไม่ยืนยันจำนวนสัตว์หรือความปลอดภัย; เป้าหมายเชื้อไม่ยืนยันการติดเชื้อจริง ผลโมเดลวัดการเลียนแบบสูตรจำลอง ไม่ใช่ความแม่นยำภาคสนาม ไม่ใช้เลือกวัคซีน ยา หรือระยะจับปลา/หมึกที่ปลอดภัย\n\nนำข้อมูลจริงมาแทนได้ด้วย schema เดียวกัน แต่ต้องตรวจหน่วย วิธีตรวจ วันขาดหาย และ outcome ที่เหมาะสมก่อนฝึกใหม่ ไม่ควรคงฉลากจากสูตรจำลองแล้วเรียกว่า validation ภาคสนาม\n')
            gr.File(str(OUT/'synthetic_daily.csv'),label='ดาวน์โหลดข้อมูลจำลอง / schema CSV',interactive=False)
        with gr.Tab('02  Forecast & Action · คาดการณ์และลงมือ',id='forecast'):
            with gr.Column(elem_classes=['ow-panel']):
                gr.HTML(section('30-day scenario forecast','ค่าทำนายแบบมีเงื่อนไขตาม inputs สมมติ · ไม่ใช่พยากรณ์อากาศ','CONDITIONAL SCENARIO'))
                with gr.Row():
                    fsite=gr.Dropdown(IDS,value=DEFAULT,label='พื้นที่สำหรับการคาดการณ์และภารกิจ')
                    fmodel=gr.Dropdown(MODEL_NAMES,value=MODEL_NAMES[0],label='โมเดล')
                    scenario=gr.Dropdown(SCENARIOS,value=SCENARIOS[0],label='สมมติฐานอนาคต')
                forecast_button=gr.Button('สร้างสถานการณ์ 30 วัน',variant='primary')
                forecast_plot=gr.Plot(forecast0[0],show_label=False)
                forecast_note=gr.Markdown(forecast0[2])
                with gr.Accordion('รายละเอียดรายวัน / ฝนหน่วงเวลา / first-rain',open=False):
                    forecast_table=gr.Dataframe(forecast0[1],interactive=False)
                f_inputs=[fsite,fmodel,scenario]; f_outputs=[forecast_plot,forecast_table,forecast_note]
                forecast_button.click(forecast_view,f_inputs,f_outputs,api_name='forecast')
                for c in f_inputs:
                    c.change(forecast_view,f_inputs,f_outputs,api_name=False)
            with gr.Row(equal_height=False):
                with gr.Column(scale=4,min_width=300,elem_classes=['ow-panel']):
                    gr.HTML(section('Model evaluation','เปรียบเทียบ predictors และช่วงเวลาชุดเดียวกัน','TIME SPLIT'))
                    gr.Plot(performance_plot(),show_label=False)
                with gr.Column(scale=6,min_width=320,elem_classes=['ow-panel']):
                    gr.Markdown(f'### ผลทดสอบคะแนนรายวันตามลำดับเวลา\nTrain {TRAIN.date.min():%d/%m/%Y}–{TRAIN.date.max():%d/%m/%Y}; Test {TEST.date.min():%d/%m/%Y}–{TEST.date.max():%d/%m/%Y}\n\nผลรวม 12 จุด ใช้ input ของวันทดสอบ ไม่ใช่ backtest ความแม่นยำพยากรณ์ 30 วัน หรือ validation ภาคสนาม')
                    gr.Dataframe(METRICS.round(3),interactive=False)
            with gr.Column(elem_classes=['ow-panel']):
                gr.HTML(section('Community action workspace','ข้อเสนอแนะตามบทบาท · อ้างอิงคะแนนปัจจุบันจำลอง','7 ROLES / 4 TASKS'))
                role=gr.Dropdown(ROLES,value=ROLES[0],label='บทบาทของคุณ')
                task_state=gr.State(context0)
                task_md=gr.Markdown(task_summary(context0))
                heads,responses,images=[],[],[]
                for start in [0,2]:
                    with gr.Row():
                        for i in range(start,start+2):
                            with gr.Column(min_width=280,elem_classes=['ow-task']):
                                heads.append(gr.Markdown(task_headers(context0)[i]))
                                responses.append(gr.Textbox(label=f'คำตอบภารกิจ {i+1}',lines=3,max_lines=6))
                                images.append(gr.Image(label=f'รูปภารกิจ {i+1} (ไม่บังคับ)',sources=['upload'],type='filepath',format='png',height=150))
                gr.Markdown(ACTION_NOTE)
                submit=gr.Button('ส่งคำตอบทั้ง 4 ภารกิจ',variant='primary')
                confirmation=gr.Markdown()
                receipt=gr.File(label='ใบรับการส่งครั้งนี้ (CSV)',interactive=False)
                reset_outputs=[task_state,task_md,*heads,*responses,*images,confirmation,receipt]
                fsite.change(reset_tasks,[fsite,role],reset_outputs,api_name=False)
                role.change(reset_tasks,[fsite,role],reset_outputs,api_name=False)
                submit.click(save_tasks,[task_state,fsite,role,*responses,*images],[confirmation,receipt],api_name=False)
                gr.Markdown('คำตอบและรูปเก็บชั่วคราวใน runtime ภายใต้ onewaterai_outputs; ไม่ใส่ข้อมูลส่วนบุคคล ดาวน์โหลดใบรับก่อนปิด Colab การส่งงานไม่เปลี่ยนสถานะคุณภาพน้ำ')
        with gr.Tab('03  Risk Simulator · ทดลองประเมิน',id='simulator'):
            gr.HTML('''<div class="ow-flow"><div><small>01 · SOURCE</small><strong>เกาะสีชัง B01–B12</strong></div><div><small>02 · DATA</small><strong>สิ่งแวดล้อม + MST + eDNA</strong></div><div><small>03 · MODEL</small><strong>Ridge / MLP 32 → 16 → 1</strong></div><div><small>04 · RISK SCORE</small><strong>คะแนนสาธิต 0–100</strong></div><div><small>05 · ACTION</small><strong>งานตามกฎที่อธิบายได้</strong></div></div>''')
            with gr.Row(equal_height=False):
                with gr.Column(scale=6,min_width=320,elem_classes=['ow-panel']):
                    gr.HTML(section('1. เลือกจุดและปรับ input','ตัวชี้วัดชีวภาพทุกตัวเป็นดัชนีจำลอง 0–100','EDITABLE INPUTS'))
                    with gr.Row():
                        ssite=gr.Dropdown(IDS,value=DEFAULT,label='จุดติดตาม')
                        smodel=gr.Dropdown(MODEL_NAMES,value=MODEL_NAMES[0],label='โมเดลที่ฝึกแล้ว')
                    gr.HTML('<div class="ow-action"><strong>Ko Sichang / เกาะสีชัง</strong>B01–B12 เป็นรหัสจุดสาธิต · พิกัดจำลอง ไม่ใช่สถานีตรวจยืนยัน<br>ดู GIS และอัปโหลดชั้นข้อมูลจริงได้ในหน้า Dashboard</div>')
                    gr.Markdown('### ENVIRONMENTAL · ค่าจำลอง')
                    edits=[]
                    for start in [0,2]:
                        with gr.Row():
                            for name,label in zip(ENV[start:start+2],ENV_LABELS[start:start+2]):
                                lo,hi=RANGES[name]
                                edits.append(gr.Slider(lo,hi,value=float(LATEST.loc[DEFAULT,name]),step=.01,label=label))
                    for title,cols in GROUPS:
                        with gr.Accordion(title,open=False):
                            for start in range(0,len(cols),2):
                                with gr.Row():
                                    for name in cols[start:start+2]:
                                        edits.append(gr.Number(float(LATEST.loc[DEFAULT,name]),label=name,minimum=0,maximum=100,precision=2))
                    lag_md=gr.Markdown(load_sim_site(DEFAULT)[-1])
                    assess_button=gr.Button('ประเมินความเสี่ยง · Run assessment',variant='primary')
                    reset_button=gr.Button('คืนค่าล่าสุดของจุดที่เลือก · Reset inputs')
                with gr.Column(scale=5,min_width=320,elem_classes=['ow-panel']):
                    evaluation_state=gr.State(input_fingerprint(DEFAULT,MODEL_NAMES[0],[float(LATEST.loc[DEFAULT,k]) for k in EDIT]))
                    result=gr.HTML(sim0[0])
                    explanation=gr.Plot(sim0[1],show_label=False)
                    sim_note=gr.Markdown(sim0[2])
                    gr.Markdown('Sensitivity คือผลต่างจากการแทนทีละตัวด้วยค่าเฉลี่ย train ไม่ใช่ causal effect หรือ feature importance ทั่วไป')
            sim_outputs=[result,explanation,sim_note,evaluation_state]
            assess_button.click(simulator_assess,[ssite,smodel,*edits],sim_outputs,api_name='simulate')
            ssite.change(reset_simulator,[ssite,smodel],[*edits,lag_md,*sim_outputs],api_name=False)
            reset_button.click(reset_simulator,[ssite,smodel],[*edits,lag_md,*sim_outputs],api_name=False)
            for c in [smodel,*edits]:
                c.input(pending_result,[evaluation_state,ssite,smodel,*edits],result,queue=False,api_name=False)
            gr.Markdown('เปลี่ยนโมเดลหรือ input แล้วกดประเมิน · เปลี่ยนจุดจะคืนค่า input และคำนวณใหม่ · ผลประเมินไม่ใช่คำยืนยันความปลอดภัย')
    gr.HTML('<footer class="ow-footer"><span>OneWaterAI · Ko Sichang environmental health prototype</span><span>SYNTHETIC DATA · UNVALIDATED MODELS · TEMPORARY COLAB STORAGE</span></footer>')

demo.queue(default_concurrency_limit=2,max_size=24)


def launch_app(share=True, **kwargs):
    """Colab: public temporary URL, light mode. Local tests use share=False."""
    app, local_url, share_url = demo.launch(share=share,show_error=True,ssr_mode=False,
        max_file_size='100mb',inbrowser=False,**kwargs)
    if share_url:
        print('Share this temporary OneWaterAI URL:',share_url+'?__theme=light')
        print('Keep the Colab runtime running. Download receipts before ending the session.')
    return app, local_url, share_url


if __name__ == '__main__' and os.environ.get('ONEWATERAI_NO_LAUNCH') != '1':
    launch_app(share=True)
